# SAM Prompt Ambiguity

Which region does the point select? Compare the three candidate masks.

Run the setup cells, then change the input in **Demo code**. All packages, models and data stay in the Colab runtime. For large models, select a GPU runtime. Free GPU availability varies.

Source: https://huggingface.co/facebook/sam-vit-base

## Runtime setup

In [ ]:
# Packages are installed in the Colab runtime, not on your computer.
%pip -q install "matplotlib>=3.9,<3.12" "scikit-learn==1.6.1" "Pillow" "requests" "tqdm" "transformers==4.57.1" "huggingface-hub==0.36.2" "accelerate" "safetensors"
import os
os.environ["TABPFN_DISABLE_TELEMETRY"] = "1"
os.environ["WANDB_MODE"] = "disabled"
import torch
print("Runtime:", "GPU " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
import os, random, requests, tarfile, shutil
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn.functional as F
from transformers import (
    AutoProcessor, AutoModelForZeroShotObjectDetection,
    SamModel, SamProcessor,
)
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Using device: {device}")

In [ ]:
GDINO_ID = "IDEA-Research/grounding-dino-tiny"
print(f"📦 Grounding DINO: {GDINO_ID}")
gdino_processor = AutoProcessor.from_pretrained(GDINO_ID)
gdino_model = AutoModelForZeroShotObjectDetection.from_pretrained(GDINO_ID).to(device)
gdino_model.eval()
gdino_params = sum(p.numel() for p in gdino_model.parameters())
print(f"   ✅ Done! ({gdino_params/1e6:.1f}M params)")

In [ ]:
SAM_ID = "facebook/sam-vit-base"
print(f"📦 SAM: {SAM_ID}")
sam_processor = SamProcessor.from_pretrained(SAM_ID)
sam_model = SamModel.from_pretrained(SAM_ID).to(device)
sam_model.eval()
sam_params = sum(p.numel() for p in sam_model.parameters())
print(f"   ✅ Done! ({sam_params/1e6:.1f}M params)")
print(f"\n📊 Total: {(gdino_params + sam_params)/1e6:.1f}M params")

## Input images

Fixed Oxford-IIIT Pet classroom subset. Source: https://www.robots.ox.ac.uk/~vgg/data/pets/

In [ ]:
from pathlib import Path
import urllib.request, zipfile

IMG_DIR = "./data"
NUM_IMAGES_PER_CLASS = 50
batch_size = 16
if not all(len(list((Path(IMG_DIR) / label).glob("*.jpg"))) == 50 for label in ["dog", "cat"]):
    url = "https://moai.postech.ac.kr/teaching/27s-programming-for-ax/labs/colab/oxford-pets-classroom.zip"
    archive, _ = urllib.request.urlretrieve(url)
    with zipfile.ZipFile(archive) as bundle:
        bundle.extractall(".")
image_paths, image_labels = [], []
for label in ["dog", "cat"]:
    paths = sorted((Path(IMG_DIR) / label).glob("*.jpg"))
    image_paths.extend(str(path) for path in paths)
    image_labels.extend([label] * len(paths))
print(f"Classroom subset: {len(image_paths)} images")


In [ ]:
import hashlib
_detection_cache = {}
_sam_embedding_cache = {}

@torch.no_grad()
def detect_objects(image, text_labels, box_threshold=0.3, text_threshold=0.25):
    """
    Use Grounding DINO to detect objects matching the text in an image.

    Args:
        image: PIL Image
        text_labels: list of str (e.g., ["dog", "cat"])
    Returns:
        boxes (N,4), scores (N,), labels list[str]
    """
    key = (hashlib.sha256(image.tobytes()).hexdigest(), image.size, tuple(text_labels))
    if key not in _detection_cache:
        inputs = gdino_processor(images=image, text=[text_labels], return_tensors="pt").to(device)
        _detection_cache[key] = gdino_model(**inputs)
    outputs = _detection_cache[key]
    results = gdino_processor.post_process_grounded_object_detection(
        outputs, threshold=box_threshold, text_threshold=text_threshold,
        target_sizes=[(image.height, image.width)]
    )
    r = results[0]
    return r['boxes'].cpu().numpy(), r['scores'].cpu().numpy(), r['text_labels']

print("✅ detect_objects defined")

In [ ]:
def draw_detections(image, boxes, scores, labels, title='', ax=None):
    if ax is None: fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    ax.imshow(image)
    unique = list(set(labels))
    cmap = plt.cm.Set2
    cdict = {l: cmap(i / max(len(unique), 1)) for i, l in enumerate(unique)}
    for box, score, label in zip(boxes, scores, labels):
        x1, y1, x2, y2 = box
        c = cdict[label]
        rect = mpatches.FancyBboxPatch((x1,y1), x2-x1, y2-y1, linewidth=3,
            edgecolor=c, facecolor='none', boxstyle='round,pad=0')
        ax.add_patch(rect)
        ax.text(x1, y1-8, f'{label}: {score:.2f}', fontsize=11, fontweight='bold',
                color='white', bbox=dict(boxstyle='round,pad=0.3', facecolor=c, edgecolor='none', alpha=0.85))
    ax.set_title(title, fontsize=13, fontweight='bold'); ax.axis('off')
    return ax

print("✅ draw_detections defined")

In [ ]:
@torch.no_grad()
def cached_sam_inputs(image, **prompts):
    """Reuse the frozen image encoder when only a prompt changes."""
    key = (hashlib.sha256(image.tobytes()).hexdigest(), image.size)
    inputs = sam_processor(images=image, return_tensors="pt", **prompts).to(device)
    pixels = inputs.pop("pixel_values")
    if key not in _sam_embedding_cache:
        _sam_embedding_cache[key] = sam_model.get_image_embeddings(pixels)
    inputs["image_embeddings"] = _sam_embedding_cache[key]
    return inputs

@torch.no_grad()
def segment_with_boxes(image, boxes):
    """
    Pass bounding boxes to SAM as prompts to generate segmentation masks.
    Process boxes one at a time to ensure shape stability.

    Args:
        image: PIL Image
        boxes: numpy array (N, 4) [x1, y1, x2, y2]
    Returns:
        masks: list of (H, W) bool numpy arrays
        iou_scores: list of float
    """
    if len(boxes) == 0:
        return [], []

    best_masks = []
    best_scores = []

    for box in boxes:
        # SAM processor expects: input_boxes = [[[x1, y1, x2, y2]]]
        input_boxes = [[box.tolist()]]
        inputs = cached_sam_inputs(image, input_boxes=input_boxes)

        outputs = sam_model(**inputs)

        # post_process_masks: restore to original image size
        masks_list = sam_processor.image_processor.post_process_masks(
            outputs.pred_masks.cpu(),
            inputs["original_sizes"].cpu(),
            inputs["reshaped_input_sizes"].cpu()
        )
        iou = outputs.iou_scores.cpu().numpy()  # (1, 1, 3)

        # masks_list[0] shape: (1, 3, H, W) → squeeze → (3, H, W)
        mask_3 = masks_list[0].squeeze(0)   # (3, H, W)
        score_3 = iou[0, 0]                  # (3,)

        best_idx = score_3.argmax()
        best_masks.append(mask_3[best_idx].numpy().astype(bool))
        best_scores.append(float(score_3[best_idx]))

    return best_masks, best_scores


@torch.no_grad()
def segment_with_points(image, points, point_labels=None):
    """Pass point prompts to SAM to generate a segmentation mask."""
    if point_labels is None:
        point_labels = [1] * len(points)

    inputs = cached_sam_inputs(image, input_points=[[points]], input_labels=[[point_labels]])

    outputs = sam_model(**inputs)
    masks_list = sam_processor.image_processor.post_process_masks(
        outputs.pred_masks.cpu(),
        inputs["original_sizes"].cpu(),
        inputs["reshaped_input_sizes"].cpu()
    )
    iou = outputs.iou_scores.cpu().numpy()
    mask_3 = masks_list[0].squeeze(0)  # (3, H, W)
    score_3 = iou[0, 0]                # (3,)
    best_idx = score_3.argmax()
    return mask_3[best_idx].numpy().astype(bool), float(score_3[best_idx])

print("✅ SAM segmentation functions defined")

## Demo code

Which region does the point select? Compare the three candidate masks.

In [ ]:
image = Image.open(image_paths[1]).convert("RGB")
point = [[image.width // 2, image.height // 2]]
mask, predicted_quality = segment_with_points(image, point)
plt.imshow(image); plt.imshow(mask, alpha=0.4)
plt.show()
print("Predicted mask quality:", predicted_quality)

In [ ]:
sample_img = Image.open(image_paths[1]).convert('RGB')
w, h = sample_img.size
center_point = [[w // 2, h // 2]]

mask, iou = segment_with_points(sample_img, center_point)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('SAM — Point Prompt Segmentation', fontsize=16, fontweight='bold')

axes[0].imshow(sample_img)
axes[0].scatter([center_point[0][0]], [center_point[0][1]],
               c='red', s=200, marker='*', zorder=5, edgecolors='white', linewidth=2)
axes[0].set_title('Input: Image + Point'); axes[0].axis('off')

axes[1].imshow(mask, cmap='coolwarm')
axes[1].set_title(f'Output: Mask (predicted quality: {iou:.3f})'); axes[1].axis('off')

axes[2].imshow(sample_img)
overlay = np.zeros((*mask.shape, 4)); overlay[mask] = [0, 0.7, 1, 0.5]
axes[2].imshow(overlay)
axes[2].scatter([center_point[0][0]], [center_point[0][1]],
               c='red', s=200, marker='*', zorder=5, edgecolors='white', linewidth=2)
axes[2].set_title('Overlay'); axes[2].axis('off')

plt.tight_layout(); plt.show()

In [ ]:
sample_img = Image.open(image_paths[0]).convert('RGB')
boxes, scores, labels = detect_objects(sample_img, ["dog"], box_threshold=0.3)

if len(boxes) > 0:
    single_box = [[boxes[0].tolist()]]
    inputs = sam_processor(images=sample_img, input_boxes=single_box, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = sam_model(**inputs)

    masks_list = sam_processor.image_processor.post_process_masks(
        outputs.pred_masks.cpu(),
        inputs["original_sizes"].cpu(),
        inputs["reshaped_input_sizes"].cpu()
    )
    iou_scores = outputs.iou_scores.cpu().numpy()  # (1, 1, 3)
    three_masks = masks_list[0].squeeze(0)          # (3, H, W)
    scores_3 = iou_scores[0, 0]                     # (3,)

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    fig.suptitle('🎭 SAM — One Box → 3 masks (different granularities)', fontsize=14, fontweight='bold')

    axes[0].imshow(sample_img)
    x1, y1, x2, y2 = boxes[0]
    rect = mpatches.Rectangle((x1,y1), x2-x1, y2-y1, linewidth=3, edgecolor='lime', facecolor='none')
    axes[0].add_patch(rect)
    axes[0].set_title('Input Box'); axes[0].axis('off')

    mask_names = ['Candidate 1', 'Candidate 2', 'Candidate 3']
    cs = [[1,0.3,0.3,0.5], [0.3,1,0.3,0.5], [0.3,0.3,1,0.5]]
    for i in range(3):
        m = three_masks[i].numpy()
        axes[i+1].imshow(sample_img)
        ov = np.zeros((*m.shape, 4)); ov[m > 0] = cs[i]
        axes[i+1].imshow(ov)
        star = ' ⭐' if i == scores_3.argmax() else ''
        axes[i+1].set_title(f'{mask_names[i]}\nPredicted quality: {scores_3[i]:.3f}{star}')
        axes[i+1].axis('off')

    plt.tight_layout(); plt.show()
    print("💡 SAM produces 3 candidate masks to handle ambiguity.")
    print("   The mask with the highest IoU score (⭐) is selected automatically.")
else:
    print("No boxes detected. Try lowering the threshold.")

## Interpretation

SAM predicts mask quality; the displayed score is not measured ground-truth IoU.

Compare the output with the evidence in the input. A generated answer can be wrong.